# VideoDB Indexing V2: Custom Index

Index your **own** timestamped records — chapters, moments, edit decisions, model outputs, anything with a `start` and `end` — without running a built-in analyzer.

A custom index answers: **"I already have structured segments for this video; make them searchable, queryable, and aggregatable."**

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/preview/guides/indexing-v2/indexing/custom_index.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Install dependencies

In [ ]:
!pip install -q videodb python-dotenv

## 2. Connect to VideoDB

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()

if not os.getenv("VIDEO_DB_API_KEY"):
    os.environ["VIDEO_DB_API_KEY"] = getpass("Enter your VideoDB API key: ")

conn = connect(api_key=os.environ["VIDEO_DB_API_KEY"])
print("Connected to VideoDB")

## 3. Choose a video

Custom records reference a video's timeline (`start` / `end` in seconds), so a custom index still attaches to a video. By default this uploads the sample clip; swap in an existing video below.

In [ ]:
VIDEO_URL = "https://www.youtube.com/watch?v=vVlEVRKv4is"  # Silicon Valley - Gilfoyle is free for hire

collection = conn.get_collection()
video = collection.upload(VIDEO_URL)

# To use an existing video instead, comment the upload line above and uncomment these:
# VIDEO_ID = "m-..."
# video = collection.get_video(VIDEO_ID)

print("Collection:", collection.id)
print("Video:", video.id)

## 4. What is a custom index?

Most indexes are built from an **understanding artifact** (`source=analyzer`). A **custom index** is built from a plain list of records you provide as the `source`:

- Each record is a dict with a `start` and `end` (seconds on the video timeline).
- Every other key becomes a **data field** you can index — `scene_id` is generated for you if you don't supply one.
- `use_for` and `fields` work exactly as they do for analyzer indexes: declare capabilities and put fields into groups (`semantic`, `filter`, `aggregate`, `sort`), or omit `fields` and let VideoDB derive them by shape.

Use it for chapters, highlights, human annotations, or outputs from your own models — anything already segmented in time.

## 5. Define your records

In [ ]:
# Your own timestamped segments. Only `start` and `end` are required;
# everything else (summary, chapter, topic, importance) becomes an indexable field.
custom_records = [
    {"start": 0.0,  "end": 8.0,  "chapter": "Setup",       "topic": "hiring",      "importance": 3,
     "summary": "Gilfoyle announces he is free for hire and open to offers."},
    {"start": 8.0,  "end": 20.0, "chapter": "Negotiation", "topic": "negotiation", "importance": 5,
     "summary": "The team debates compensation and leverage over a rival company."},
    {"start": 20.0, "end": 32.0, "chapter": "Decision",    "topic": "hiring",      "importance": 4,
     "summary": "A decision is reached and the terms are laid out plainly."},
]

# The SDK wraps a list source as {"scenes": custom_records} for you.
len(custom_records)

## 6. Create the custom index

Pass the list as `source`. Here the prose `summary` is embedded for semantic search, while `chapter`/`topic` are filterable and `topic`/`importance` are aggregatable.

In [ ]:
from datetime import datetime

custom_index_name = f"chapters_{datetime.utcnow().strftime('%Y%m%d%H%M%S')}"

custom_index = video.index(
    name=custom_index_name,
    source=custom_records,
    use_for=["semantic", "query", "aggregate"],
    fields={
        "semantic": ["summary"],
        "filter": ["chapter", "topic"],
        "aggregate": ["topic", "importance"],
        "sort": ["importance"],
    },
)
custom_index

## 7. Wait and inspect

In [ ]:
custom_index.wait_until_complete(timeout=900, poll_interval=10)
print("status:", custom_index.status)
if not custom_index.is_successful:
    print("build failed:", custom_index.error)

print("fields:", custom_index.fields)
for field, schema in custom_index.field_schema.items():
    print(" ", field, schema.type, schema.groups)

## 8. Semantic search

Search the `summary` field by meaning.

In [ ]:
results = video.semantic_search(
    query="deciding whether to accept a job offer",
    index_names=[custom_index_name],
    top_k=3,
    return_fields="all",
)
results

## 9. Structured query and aggregate

Filter and group by the fields you declared.

In [ ]:
# exact filter on a custom field
hiring = video.query(
    index_name=custom_index_name,
    filter={"field": "topic", "op": "==", "value": "hiring"},
    return_fields="all",
)
print("hiring segments:", len(hiring))

# count segments per topic
topic_counts = video.aggregate(
    index_name=custom_index_name,
    group_by="topic",
    metric="count",
)
print("topic counts:", topic_counts)

## 10. Cleanup

In [ ]:
# custom_index.delete()   # or: video.delete_index(index_id=custom_index.index_id)